# Análise da Baseline do DeepLog (Fase 2.3)

Este notebook faz parte da pesquisa de mestrado **Logscan-XDP**. O objetivo deste documento é analisar de forma estruturada e científica os resultados obtidos no experimento da **Fase 2 (Baseline)**. 

Aqui estabelecemos o marco zero (baseline) do modelo de Inteligência Artificial **DeepLog LSTM** puro rodando no espaço de usuário (User Space) sobre o dataset de referência **HDFS completo**.

## 1. Carregamento dos Dados de Baseline

Vamos importar a biblioteca `pandas` e carregar o arquivo CSV `results/data/baseline.csv` que consolidou os dados obtidos durante o treinamento de 15 épocas e inferência completa de teste.

In [ ]:
import os
import pandas as pd

# Caminho do CSV de baseline
csv_path = "../data/baseline.csv"

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    # Exibe a tabela estruturada com estilo
    display(df.style.set_caption("Métricas Experimentais da Baseline - DeepLog HDFS (CUDA)").format(precision=3))
else:
    print(f"Erro: O arquivo '{csv_path}' não foi localizado. Certifique-se de executar o pipeline de coleta primeiro.")

## 2. Visualização Científica dos Resultados

Para incorporar os gráficos diretamente à dissertação de mestrado e artigos científicos, utilizaremos a biblioteca `matplotlib` e `seaborn` para gerar e renderizar as figuras de alta fidelidade estruturadas em nosso script automatizado.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configura o tema estético profissional
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 12,
    'axes.labelsize': 13,
    'axes.titlesize': 14,
    'savefig.dpi': 300
})

print("Configuração estética científica carregada com sucesso!")

### A. Métricas de Acurácia da Rede Neural LSTM

O gráfico abaixo exibe as métricas de Precision (Precisão), Recall (Sensibilidade) e F1-Score do DeepLog treinado.

In [ ]:
plt.figure(figsize=(7, 6))
metrics = {
    'Precision': df['Precision'].iloc[0],
    'Recall': df['Recall'].iloc[0],
    'F1-Score': df['F1_Score'].iloc[0]
}

colors = ['#1b9e77', '#377eb8', '#e41a1c']
bars = plt.bar(metrics.keys(), metrics.values(), color=colors, width=0.5, edgecolor='black', linewidth=0.8)

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2.0, 
        height + 2,
        f'{height:.3f}%', 
        ha='center', 
        va='bottom', 
        fontweight='bold'
    )

plt.ylabel('Porcentagem (%)', fontweight='bold')
plt.title('Desempenho de Classificação do DeepLog (Baseline)', fontweight='bold', pad=15)
plt.ylim(0, 110)
plt.tight_layout()
plt.show()

### B. Comparação do Custo Temporal: Treino vs Inferência

Este gráfico evidencia a disparidade temporal entre o treinamento acelerado por GPU CUDA e a inferência iterativa efetuada em espaço de usuário (CPU).

In [ ]:
plt.figure(figsize=(7, 6))
times = {
    'Treinamento\n(GPU CUDA)': df['TrainingTimeSeconds'].iloc[0],
    'Inferência completa\n(User Space CPU)': df['PredictionTimeSeconds'].iloc[0]
}

colors = ['#4daf4a', '#ff7f00']
bars = plt.bar(times.keys(), times.values(), color=colors, width=0.4, edgecolor='black', linewidth=0.8)

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2.0, 
        height + (max(times.values()) * 0.02),
        f'{height:.2f}s', 
        ha='center', 
        va='bottom', 
        fontweight='bold'
    )

plt.ylabel('Tempo em Segundos (s)', fontweight='bold')
plt.title('Custo Temporal: Treino GPU vs Inferência CPU', fontweight='bold', pad=15)
plt.ylim(0, max(times.values()) * 1.15)
plt.tight_layout()
plt.show()

## 3. Análise Crítica dos Resultados e Discussão de Artigo

### 1. Robustez do DeepLog no HDFS
O modelo LSTM apresentou uma taxa de **Recall de 93.919%**, garantindo que quase todas as anomalias reais de blocos de execução do Hadoop sejam devidamente capturadas. A **Precisão de 86.425%** denota que, mesmo sendo um classificador puramente não-supervisionado que aprende apenas o fluxo saudável, o número de alarmes falsos é perfeitamente aceitável academicamente.

### 2. O Gargalo Crítico do Espaço de Usuário
O dado mais revelador do experimento está no custo de inferência: enquanto o treinamento durou apenas **11.10 segundos** graças à paralelização na GPU CUDA, a **inferência completa em espaço de usuário demorou 74.60 segundos**.

Em cenários de produção sob ataque, um volume massivo de logs normais (hotspots de redundância de I/O) causará saturação da CPU do host. Como a predição exige avaliar a probabilidade de transição de cada linha, a rede neural engasgará a fila de processamento.

### 3. A Oportunidade do eBPF/XDP (Fases 3 e 4)
Esta baseline comprova a hipótese da tese: **o espaço de usuário é ineficiente para a triagem primária sob estresse**.

Ao desenvolvermos o filtro de alta velocidade em eBPF na **Fase 3** (programado diretamente no Kernel ou descarregado na SmartNIC via XDP), seremos capazes de barrar os blocos repetitivos e saudáveis antes mesmo de consumirem CPU no host ou disco. Isso permitirá reduzir o tempo de inferência de 74.60s para frações de segundo, poupando a CPU principal do servidor sem degradar o F1-Score do DeepLog.